In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import kagglehub
import os

import matplotlib.pyplot as plt
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.model_selection import train_test_split, KFold
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, StandardScaler

import warnings
warnings.filterwarnings('ignore')

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
# Task 1: Write your code here:
FoodDelivery_path = os.path.join(path, 'Q1_data.csv')
df_FoodDelivery = pd.read_csv(FoodDelivery_path)

In [ ]:
# Task 2: Write your code here:
print(f"Shape: {df_FoodDelivery.shape}")
df_FoodDelivery.head()

In [ ]:
# Task 3: Write your code here:
df_FoodDelivery.info()

In [ ]:
# Task 4: Write your code here:
df_FoodDelivery.describe()

In [ ]:
# Task 5: Write your code here:
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 5))
plt.hist(df_FoodDelivery['Delivery_Time'].dropna(), bins=30, edgecolor='black')
plt.title('Food Delivery Distribution')
plt.xlabel('Orders')
plt.ylabel('Time')
plt.show()

In [ ]:
# Task 1: Write your code here:
df_FoodDelivery.drop(columns=['Order_ID'])

In [ ]:
# Task 2: Write your code here:

# Define stat columns
print("Missing values:")
print(df_FoodDelivery.isnull().sum())

stat_cols = ['Weather', 'Traffic_Level', 'Time_of_Day', 'Delivery_Time']

# Drop rows with missing stat values
df_clean = df_FoodDelivery.dropna(subset=stat_cols).copy()
print(f"Shape after cleaning: {df_clean.shape}")



In [ ]:
# Task 3: Write your code here:
print("Checking for duplicate rows...")
duplicate_rows = df_FoodDelivery.duplicated().sum()
if duplicate_rows > 0:
    print(f"Found {duplicate_rows} duplicate rows. Removing them...")
    df_FoodDelivery.drop_duplicates(inplace=True)
    print("Duplicate rows removed.")
else:
    print("No duplicate rows found.")

In [ ]:
# Task 4: Write your code here:

# Encode type columns
le = LabelEncoder()
df_clean['Weather'] = le.fit_transform(df_clean['Weather'])
df_clean['Traffic_Level'] = le.fit_transform(df_clean['Traffic_Level'])
df_clean['Time_of_Day'] = le.fit_transform(df_clean['Time_of_Day'])
df_clean['Vehicle_Type'] = le.fit_transform(df_clean['Vehicle_Type'])

In [ ]:
# Task 5: Write your code here:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
# Task 6: Write your code here:
print("\nValue counts for the target variable 'Delivery_Time':")
class_distribution = df_FoodDelivery['Delivery_Time'].value_counts()
print(class_distribution)

plt.figure(figsize=(6, 4))
sns.countplot(data=df_clean, x='Delivery_Time')
plt.title('Distribution of Target Variable (Delivery_Time)')
plt.xlabel('Orders')
plt.ylabel('Time')
plt.show()

In [ ]:
# Task 1: Write your code here:
feature_cols = ['Distance_km', 'Weather', 'Traffic_Level', 'Time_of_Day','Vehicle_Type', 'Preparation_Time_min','Courier_Experience_yrs']


X = df_clean[feature_cols]
y = df_clean['Delivery_Time']

# Stratified split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=None)


In [ ]:
# Task 2,3,4,5: Write your code here:

# Train Random Forest Regressor
model = RandomForestRegressor(n_estimators=100, max_depth=20, random_state=42, n_jobs=-1)
model.fit(X_train_scaled, y_train)
print("Model trained!")

kfold = KFold(n_splits=5, shuffle=True, random_state=42)

mae_scores = []
rmse_scores = []

for train_idx, val_idx in kfold.split(X_train_scaled):
    X_fold_train, X_fold_val = X_train_scaled[train_idx], X_train_scaled[val_idx]
    y_fold_train, y_fold_val = y_train.iloc[train_idx], y_train.iloc[val_idx]

    # Train and predict
    model.fit(X_fold_train, y_fold_train)
    y_fold_pred = model.predict(X_fold_val)

    # Calculate metrics
    mae_scores.append(mean_absolute_error(y_fold_val, y_fold_pred))
    rmse_scores.append(np.sqrt(mean_squared_error(y_fold_val, y_fold_pred)))

mae_scores = np.array(mae_scores)
rmse_scores = np.array(rmse_scores)

print(f"5-Fold CV Results:")
print(f"MAE:  {mae_scores.mean():,.2f}")

In [ ]:
# Task 1: Write your code here:
# Feature importance
feature_importance = pd.DataFrame({
    'feature': feature_cols,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

plt.figure(figsize=(10, 6))
plt.barh(feature_importance['feature'], feature_importance['importance'])
plt.xlabel('Importance')
plt.title('Feature Importance')
plt.gca().invert_yaxis()
plt.show()

In [ ]:
# Task 2: Write your code here:
# Price distribution (target variable)
plt.figure(figsize=(10, 5))
plt.hist(df_FoodDelivery['Delivery_Time'].dropna(), bins=50, edgecolor='black')
plt.title('Delivery_Time Distribution')
plt.xlabel('Orders')
plt.ylabel('Time')
plt.show()

In [ ]:
# Task Bonus: Write your code here: